# Session 1

1. Python and Jupyter Notebooks
2. Loading data with Pandas
3. Cleaning the data
4. Adding Molecular Properties using RDKit
5. Data exploration
6. Fingerprints and their similarity

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install rdkit

## Python and Jupyter Notebooks

A **Jupyter notebook** combines executable code, equations, visualizations, and explanatory text into a single interactive document. It is organized into *cells*, which can be of two main types:
* Rich-text or Markdown cells (like this cell)
* Code cells

You can run (execute) each cell individually by pressing `Shift+Enter` or `Ctrl+Enter`. Once a cell has been executed, all variables, functions, and other definitions remain available to both earlier and later cells in the notebook. Thus, you need to be careful with the order in which you execute your cells!

**Python** is a widely used general-purpose high-level programming language. The term "high-level" means that the language has abstracted away most of the technical details and manages them for you (e.g. memory allocation).

In [ ]:
print(f"In this course, we will use Python v{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}, the default version in Colab.")

Python programs are built from a few basic elements:
* Variables (e.g., `x`, `y`) – store data
* Objects (e.g., `0`, `True`, `"abc"`) – numbers, boolean, text, lists, etc.
* Operators (e.g., `+`, `==`, `=`) – perform calculations, comparisons, or assignments
* Control flow – if, for, while for logic and loops
* Functions – reusable blocks of code
* Modules – libraries that extend functionality

### Variable, objects, and operators

Exploring different data types (integer, boolean, and string).

In [ ]:
# assignments
x = 1
y = 3

# multiply and assign
z = x * y

print(f"x * y = {z}")

In [ ]:
print(f"{x} * {y} = {z}")

In [ ]:
# boolean
b = True
print(f"{b=}")
print(f"The two variables {x} and {y} are the same: {x==y}")

In [ ]:
# strings
x_str = "Hello"
y_str = "CIC students"

# concatenating strings: plus-operaot on strings
s = x_str + " " + y_str

print(s)

### Lists

In [ ]:
# create a list
activities = [47, 150.4, 9.2, 42.5]

# access a list (zero-indexed)
first_element = activities[0]
last_element = activities[-1]
activities[2] = 95.3

print(f"{first_element=}, {last_element=}")

In [ ]:
# append element to list
activities.append(107.3)

print(f"{activities=}")

### Control flows

*for-loops* and *if-then-else* statements.

Calculate the mean value of a list.

In [ ]:
# iterate over list with a for-loop
sum = 0
for n in activities:
    sum += n  # short form for (sum = sum + n)

# devide it by the number of activity values (length of list)
mean_activity = sum / len(activities)

print(f"The mean of {activities} is {mean_activity}nM")

Check if your variable fulfills a condition.

In [ ]:
# conditional execution: if-then-else
if mean_activity < 100:
    print(f"Mean activity is below 100nM.")
else:
    print(f"Mean activity >= 100nM.")

### Functions

Let's write a function that converts the IC50 value in an pIC50 value.

In [ ]:
import math

# define a function
def IC50_to_pIC50(IC50_value: float) -> float:
    # pIC50 = - log_10(IC50)
    logIC50 = math.log10(IC50_value)
    return -logIC50

In [ ]:
# execute a function

# the mean_activity (IC50) is in nM, thus, to be correct, we need to convert it to M first:
# 1 nM = 10^(-9) M
# in python, we can perform the 'power' operation with **:

mean_activity_M = mean_activity * 10**(-9)

mean_activity_pIC50 = IC50_to_pIC50(mean_activity_M)

print(f"IC50 = {mean_activity}nM; pIC50 = {mean_activity_pIC50:.3f}")

### Further reading

For a more detailed intro to Python, please refer to:
* [AI in medicine](https://colab.research.google.com/github/volkamerlab/ai_in_medicine/blob/master/week1_session1_grundkonzepte.ipynb#scrollTo=TYWuwdej6U7A)
* [Python tutorial](https://docs.python.org/3/tutorial/index.html)

## Loading data with Pandas

**Pandas** is a powerful Python library for data manipulation and analysis. The [pandas](https://pandas.pydata.org/) library has two main containers of data, the `DataFrame` (2D) and the `Series` (1D).

### Reading and exploring the data

Let's start working with pandas on a list of molecules.

We will work with acticity data of EGFR extracted from [Chembl33](https://www.ebi.ac.uk/chembl/) (Check data curation [here](https://github.com/openkinome/kinodata/blob/master/EGFR_bioactivities-in-chembl/EGFR-bioactivites-in-chembl.ipynb)).

In [ ]:
import pandas as pd

# read the data from a csv file into a dataframe
egfr_df = pd.read_csv(
    "https://github.com/volkamerlab/cic_summerschool_2025/raw/refs/heads/main/data/EGFR-activities-chembl33.csv",
    index_col=0,
)

# show the first 5 rows
egfr_df.head()

**Tip**: You can check out available functionalities of a library in this Jupyter notebook, by writing the library name followed by a dot. All available functionalities will pop up for you to explore. Since there are a lot of options, you can narrow it down by writing e.g. `read` while the popup windows is up.

**Note**: If you are working in Google Colab you migth first have to enable `Show context-powered code completions` on `Tools` > `Settings` > `Editor` in order to be able to use this feature.

In [ ]:
# what are dimentions of the data?
egfr_df.shape

In [ ]:
# which columns are in the data set
print(egfr_df.columns)

We can access columns by their name and rows by the index:

In [ ]:
egfr_df["molecule_dictionary.chembl_id"][123]

### Manipulating the dataframe

We can also drop colums, we are not interested in:

In [ ]:
egfr_df = egfr_df.drop(columns=["docs.authors", "docs.year"])
egfr_df.head()

**Note:** `egfr_df.drop(columns=["docs.authors", "docs.year"])` does **not** not change the dataframe, **but** create a new one. Thus, only executing `egfr_df.drop(columns=["docs.authors", "docs.year"])` without assignment will do nothing. If you want to directly manipulate the Dataframe *in-place*, you can use the `inplace=True` option.

Or select those columns, we want to consider:

In [ ]:
egfr_activities = egfr_df[
    [
        "molecule_dictionary.chembl_id",
        "activities.standard_type",
        "activities.standard_value",
        "activities.standard_units",
        "compound_structures.canonical_smiles",
    ]
]
egfr_activities.head()

In [ ]:
egfr_activities.shape

Let's rename some of the columns:

In [ ]:
egfr_activities = egfr_activities.rename(
    columns={
        "activities.standard_type": "standard_type",
        "activities.standard_value": "standard_value",
        "activities.standard_units": "standard_units",
        "compound_structures.canonical_smiles": "canonical_smiles",
    }
)
egfr_activities

In [ ]:
egfr_activities.dtypes

## Cleaning the data
Before analyzing our dataset, we need to *clean* the data. This includes handling
1.   missing values,
2.   non-standardized (bioactivity) values,
3. and duplicates.

### Removing NaN entries

Let's first handle entries with **missing** (i.e., `None` or `NaN`) **values**:

In [ ]:
print(f"Number of entries with missing (None) values:\n{egfr_activities.isna().sum()}")

As these are only very few samples with missing SMILES, we will simply remove them:

In [ ]:
egfr_activities = egfr_activities.dropna(subset="canonical_smiles")
egfr_activities.shape

### Subselect data

Next, we will **standardize the acticity values**:

In [ ]:
egfr_activities["standard_type"].value_counts()

The standard (bioactivity) values come in different types. Without converting them—which would require additional information such as substrate concentration and the Michaelis constant (see Cheng–Prusoff)—they cannot be compared directly. Therefore, in this analysis we will **only consider pIC50** values.

In [ ]:
egfr_activities = egfr_activities[egfr_activities["standard_type"] == "pIC50"]
egfr_activities.shape

We can now drop the `standard_type` and `standard_units` column and rename the `standard_value` column:

In [ ]:
egfr_activities = egfr_activities.rename(columns={"standard_value": "pIC50"})
egfr_activities = egfr_activities.drop(columns=["standard_type", "standard_units"])
egfr_activities.head()

### Grouping data in dataframes

In molecular data from public sources, we often have the issue of having *duplicates*, i.e., molecules that have been measured several times by different groups.

In [ ]:
print(
    f"Number of duplicates: {egfr_activities['molecule_dictionary.chembl_id'].duplicated().sum()}"
)

There are different approaches, how to deal with duplicates, e.g. simply drop them, use the mean value, ....

Here, we will use the mean pIC50 for duplicates:

In [ ]:
egfr_activities_deduplicated = egfr_activities.groupby(
    "molecule_dictionary.chembl_id", as_index=False
).agg(
    {
        "molecule_dictionary.chembl_id": "first",
        "canonical_smiles": "first",
        "pIC50": "mean"
    }
)
egfr_activities_deduplicated.head()

As a sanity check, we will verify that there are no duplicate SMILES, i.e., entries with different ChEMBL IDs but identical SMILES strings:

In [ ]:
print(
    f"Number of duplicated SMILES: {egfr_activities_deduplicated['canonical_smiles'].duplicated().sum()}"
)

Since some rows were removed in this section, creating non-consecutive indices, we reset the indices:

In [ ]:
egfr_activities_deduplicated = egfr_activities_deduplicated.reset_index(drop=True)

## Adding domain specific information to the dataframe

### Working with molecular data: RDKit

**RDKit** is a powerful library for working with molecules. We will use it to calculate various molecular properties of our EGFR ligands, which will later help us explore and better understand the data.

In RDKit, we can create RDKit molecule instances from their SMILES:

In [ ]:
from rdkit import Chem

smiles_caffeine = "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"
caffeine = Chem.MolFromSmiles(smiles_caffeine)
caffeine

We can also caluclate different physiochemical properties of the molcule:

In [ ]:
from rdkit.Chem import Descriptors, rdMolDescriptors

print(f"MolWt: {Descriptors.MolWt(caffeine)}")
print(f"logP: {Descriptors.MolLogP(caffeine)}")
print(f"Num HBA: {rdMolDescriptors.CalcNumHBA(caffeine)}")
print(f"Num HBD: {rdMolDescriptors.CalcNumHBD(caffeine)}")
print(f"Num rotable bonds (RB): {rdMolDescriptors.CalcNumRotatableBonds(caffeine)}")

### Adding molecular properties to the data frame

To add the molecular weight and logP to the our EGFR compound dataset, we first create RDKit molecular instances from their SMILES:

In [ ]:
from rdkit.Chem import PandasTools

# ensure that molecules are displayed in dataframe
PandasTools.RenderImagesInAllColumns = True

PandasTools.AddMoleculeColumnToFrame(egfr_activities_deduplicated, "canonical_smiles")
egfr_activities_deduplicated

In [ ]:
egfr_activities_deduplicated["mol_weight"] = egfr_activities_deduplicated[
    "ROMol"
].apply(Descriptors.MolWt)
egfr_activities_deduplicated

In [ ]:
egfr_activities_deduplicated["logP"] = egfr_activities_deduplicated["ROMol"].apply(
    Descriptors.MolLogP
)
egfr_activities_deduplicated

## Data exploration and visualization

### Dataframe statistics

In [ ]:
egfr_activities_deduplicated.describe()

We first consider the molecular weight of the compounds:

In [ ]:
num_small_molecules = (egfr_activities_deduplicated["mol_weight"] <= 500).sum()
percent_small_molecules = 100 * num_small_molecules / len(egfr_activities_deduplicated)
print(
    f"Molecules with molecular weight <= 500 Da: {num_small_molecules} ({percent_small_molecules:.1f}%)"
)

### Plotting the data

We can also directly create a histogram using the `hist()` function on a Pandas `Series`:

In [ ]:
import matplotlib.pyplot as plt

egfr_activities_deduplicated["mol_weight"].hist()
plt.xlabel("Molecular weight [Da]")
plt.ylabel("Count")
plt.show()

A more powerful solution to plot the data are the libraries [seaborn](https://seaborn.pydata.org/) and [mathplotlib](https://matplotlib.org/).

Let's plot the distribution of pIC50 values:

In [ ]:
import seaborn as sns

sns.kdeplot(egfr_activities["pIC50"])  # density plot
sns.rugplot(egfr_activities["pIC50"])
plt.xlabel(f"pIC50")
plt.show()

We can also look at pairwise relationships of all variables:

In [ ]:
sns.pairplot(egfr_activities_deduplicated[["mol_weight", "logP", "pIC50"]])
plt.show()

## Fingerprints and their similarity
Since the molecular properties, we explored so far, do not strongly correlate with activity values, topological fingerprints are commonly used to represent molecules for machine learning models and to quantify similarity between molecules.

In this section, we will generate *Morgan fingerprints* and calculate pairwise similarity using the *Tanimoto* coefficient.

In [ ]:
from rdkit.Chem import AllChem

# RDkit provides a fingerprint generator, we first intitialize it
fpgen = AllChem.GetMorganGenerator(radius=2, fpSize=1024)

Let's explore the fingerprint of the first molecule

In [ ]:
first_fp=fpgen.GetFingerprint(
    egfr_activities_deduplicated["ROMol"][0],
)
print(first_fp)

To actually see the bits, we need to convert it to a bitstring

In [ ]:
# Convert to bit string
bitstring = first_fp.ToBitString()
print(bitstring)

Now, let's apply the fp generator to the full dataframe

In [ ]:
egfr_activities_deduplicated["morgan_fp"] = egfr_activities_deduplicated["ROMol"].apply(
    fpgen.GetFingerprint
)
egfr_activities_deduplicated

The Tanimoto similarity between the first two compounds:

In [ ]:
from rdkit.Chem import DataStructs

similarity = DataStructs.TanimotoSimilarity(
    egfr_activities_deduplicated["morgan_fp"][0],
    egfr_activities_deduplicated["morgan_fp"][1],
)

print(f"Tanimoto coefficient: {similarity:.2f}")

In addition to the similarity, we also want to display the molecues:

In [ ]:
Chem.Draw.MolsToGridImage(
    egfr_activities_deduplicated["ROMol"][:2],
    legends=list(egfr_activities_deduplicated["molecule_dictionary.chembl_id"][:2]),
)

Finally, we save the prepared dataset as a CSV file for use in the next session. Note that in Colab, this file will be lost once the session ends, thus we provide the precalculated file in the repository.

In [ ]:
if not IN_COLAB:
    egfr_activities_deduplicated.to_csv(
        "../data/EGFR-activities-prepared.csv",
        columns=[
            "pIC50",
            "molecule_dictionary.chembl_id",
            "canonical_smiles",
        ],
    )

## Exercises

### Exercise 1

In [ ]:
# Complete the function that converts the pIC50 to IC50
def pIC50_to_IC50(pIC50_value: float) -> float:
    # TODO
    return None

In [ ]:
# Use the function to add an IC50 column to the deduplicated egfr dataframe

# TODO

In [ ]:
# Plot the distributions of IC50 values

# TODO

# Would you prefer IC50 or pIC50 values for visualization? Why?

### Exercise 2

In [ ]:
# Add the following molecular properties to the deduplicated egfr dataframe: number of HBD, HBA, RB

# TODO

In [ ]:
# Complete this function that determines how many of the four Rule-of-Five (Ro5) rules are fulfilles:
# molecular weigth <= 500 D
# number of HBD <= 5
# number of HBA <= 10
# 0 <= logP <= 5

def num_ro5(mol_weight, num_HBA, num_HBD, logP) -> int:
    # TODO
    return None

In [ ]:
# Add a column to the dataframe that states how many Ro5 rules are fulfilled

# TODO

In [ ]:
# Determine how many ligands fulfill all four Ro5

# TODO

### Exercise 3 (Bonus)
Abemaciclib is an FDA-approved CDK4/6 drug. In this exercise, we aim to find the ligand in our dataset that has the highest similarity towards this CDK4/6 inhibitor (in terms of the Dice similarity of MACCS fingerprints).

In [ ]:
# Add the MACCS fingerprints of each compound to the dataframe using RDKit functionality

# TODO

egfr_activities_deduplicated["maccs_fp"] = None

In [ ]:
# Calculate the Dice similarity between each compound towards abemaciclib and save it as new column in the dataframe

abemaciclib_smiles = "CCN1CCN(Cc2ccc(Nc3ncc(F)c(-c4cc(F)c5nc(C)n(C(C)C)c5c4)n3)nc2)CC1"

# TODO

egfr_activities_deduplicated["dice_similarity"] = None

In [ ]:
# Sort the dataframe in descenting order on the 'dice_similarity', and display the 3 most similar compounds